In [ ]:
import os, sys
from pathlib import Path

print("cwd:", Path.cwd())
print("FIN_MATH_ROOT:", os.environ.get("FIN_MATH_ROOT"))
print("sys.path[0:5]:", sys.path[:5])
print("has engines in cwd?:", (Path.cwd() / "engines").is_dir())
print("has engines in FIN_MATH_ROOT?:", bool(os.environ.get("FIN_MATH_ROOT")) and (Path(os.environ["FIN_MATH_ROOT"]) / "engines").is_dir())


cwd: /content
FIN_MATH_ROOT: None
sys.path[0:5]: ['/content', '/env/python', '/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload']
has engines in cwd?: False
has engines in FIN_MATH_ROOT?: False


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [1]:
import os
import sys
from pathlib import Path

def _find_repo_root(start: Path, name: str = 'Fin_Math') -> Path:
    for path in [start, *start.parents]:
        if (path / 'engines').is_dir() and (path / 'models').is_dir():
            return path

    env_root = os.environ.get('FIN_MATH_ROOT')
    if env_root:
        path = Path(env_root).expanduser().resolve()
        if (path / 'engines').is_dir() and (path / 'models').is_dir():
            return path

    candidates = [
        Path.home() / 'Desktop' / name,
        Path.home() / 'Documents' / name,
    ]
    for path in candidates:
        if (path / 'engines').is_dir() and (path / 'models').is_dir():
            return path

    raise FileNotFoundError(
        "Could not find repo root containing 'engines' and 'models'.\n"
        "Set FIN_MATH_ROOT or update the candidates list."
    )

root = _find_repo_root(Path.cwd().resolve())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))


FileNotFoundError: Could not find repo root containing 'engines' and 'models'.
Set FIN_MATH_ROOT or update the candidates list.

In [2]:
from engines.monte_carlo import MonteCarloPricing
from models.binomial_method import BinomialPricing

In [10]:
# European option pricing: Binomial (CRR) vs plain Monte Carlo
S0, K, r, sigma, T = 100.0, 100.0, 0.05, 0.2, 1.0
steps_tree = 1000
paths = 100_000
steps_mc = 1000  # European payoff only needs terminal draw

binom = BinomialPricing(S_0=S0, K=K, r=r, sigma=sigma, T=T, steps=steps_tree)
mc = MonteCarloPricing(S_0=S0, X=K, r=r, sigma=sigma, T=T, num_paths=paths, steps=steps_mc, seed=42)

binom_call = binom.european(call=True)
binom_put = binom.european(call=False)
mc_call, mc_call_se = mc.european(call=True, antithetic=False)
mc_put, mc_put_se = mc.european(call=False, antithetic=False)

print(f"Binomial (CRR) call: {binom_call:.4f}")
print(f"MC call: {mc_call:.4f} (SE: {mc_call_se:.4f})")
print(f"Diff (MC - Binom): {mc_call - binom_call:.4f}")
print()
print(f"Binomial (CRR) put:  {binom_put:.4f}")
print(f"MC put:  {mc_put:.4f} (SE: {mc_put_se:.4f})")
print(f"Diff (MC - Binom): {mc_put - binom_put:.4f}")


Binomial (CRR) call: 10.4486
MC call: 10.4958 (SE: 0.0464)
Diff (MC - Binom): 0.0473

Binomial (CRR) put:  5.5715
MC put:  5.5218 (SE: 0.0273)
Diff (MC - Binom): -0.0497


In [11]:
# Analytical benchmark (Black-Scholes) on a scenario grid
import math
from itertools import product

def _norm_cdf(x: float) -> float:
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))

def black_scholes_price(S0: float, K: float, r: float, sigma: float, T: float, *, call: bool) -> float:
    if T <= 0.0 or sigma <= 0.0 or S0 <= 0.0 or K <= 0.0:
        raise ValueError('Inputs must be positive and T>0, sigma>0.')
    d1 = (math.log(S0 / K) + (r + 0.5 * sigma**2) * T) / (sigma * math.sqrt(T))
    d2 = d1 - sigma * math.sqrt(T)
    if call:
        return S0 * _norm_cdf(d1) - K * math.exp(-r * T) * _norm_cdf(d2)
    return K * math.exp(-r * T) * _norm_cdf(-d2) - S0 * _norm_cdf(-d1)

K = 100.0
r = 0.05
moneyness = [1.0]  # S0 / K
maturities = [0.25, 1.0]
vols = [0.15, 0.25]

paths = 50_000
steps_mc = 1
steps_tree = 1000

rows = []
for m, T, sigma in product(moneyness, maturities, vols):
    S0 = m * K
    for call in (True, False):
        bs = black_scholes_price(S0, K, r, sigma, T, call=call)
        mc = MonteCarloPricing(S_0=S0, X=K, r=r, sigma=sigma, T=T,
                               num_paths=paths, steps=steps_mc, seed=123)
        mc_price, mc_se = mc.european(call=call, antithetic=False)
        binom = BinomialPricing(S_0=S0, K=K, r=r, sigma=sigma, T=T, steps=steps_tree)
        binom_price = binom.european(call=call)

        rows.append({
            'S0': S0,
            'K': K,
            'T': T,
            'sigma': sigma,
            'call': call,
            'BS': bs,
            'MC': mc_price,
            'MC_se': mc_se,
            'Binom': binom_price,
            'MC_abs_err': abs(mc_price - bs),
            'MC_rel_err': abs(mc_price - bs) / bs,
            'Binom_abs_err': abs(binom_price - bs),
            'Binom_rel_err': abs(binom_price - bs) / bs,
        })

try:
    import pandas as pd
    df = pd.DataFrame(rows)
    pd.set_option('display.precision', 6)
    display(df)
except ImportError:
    headers = ['S0','K','T','sigma','call','BS','MC','MC_se','Binom','MC_abs_err','MC_rel_err','Binom_abs_err','Binom_rel_err']
    print(' '.join(f'{h:>12s}' for h in headers))
    for row in rows:
        print(
            f"{row['S0']:12.2f}{row['K']:12.2f}{row['T']:12.2f}{row['sigma']:12.2f}"
            f"{str(row['call']):>12s}{row['BS']:12.6f}{row['MC']:12.6f}{row['MC_se']:12.6f}"
            f"{row['Binom']:12.6f}{row['MC_abs_err']:12.6f}{row['MC_rel_err']:12.6f}"
            f"{row['Binom_abs_err']:12.6f}{row['Binom_rel_err']:12.6f}"
        )


,S0,K,T,sigma,call,BS,MC,MC_se,Binom,MC_abs_err,MC_rel_err,Binom_abs_err,Binom_rel_err
0,100.0,100.0,0.25,0.15,True,3.635070,3.646374,0.022357,3.634317,0.011305,0.003110,0.000753,0.000207
1,100.0,100.0,0.25,0.15,False,2.392850,2.383346,0.016759,2.392097,0.009504,0.003972,0.000753,0.000315
2,100.0,100.0,0.25,0.25,True,5.598400,5.618217,0.037048,5.597156,0.019816,0.003540,0.001244,0.000222
3,100.0,100.0,0.25,0.25,False,4.356180,4.341389,0.028327,4.354936,0.014791,0.003395,0.001244,0.000286
4,100.0,100.0,1.00,0.15,True,8.591658,8.615658,0.049902,8.590127,0.024000,0.002793,0.001532,0.000178
5,100.0,100.0,1.00,0.15,False,3.714601,3.697124,0.028079,3.713069,0.017476,0.004705,0.001532,0.000412
6,100.0,100.0,1.00,0.25,True,12.335999,12.379047,0.082880,12.333527,0.043048,0.003490,0.002472,0.000200
7,100.0,100.0,1.00,0.25,False,7.458941,7.433327,0.048541,7.456470,0.025614,0.003434,0.002472,0.000331


In [14]:
# Step-size sensitivity (European MC should be stable across steps)
S0, K, r, sigma, T = 100.0, 100.0, 0.05, 0.2, 1.0
steps_list = [1, 52, 252, 1000]
paths = 500_000

bs_call = black_scholes_price(S0, K, r, sigma, T, call=True)
print(f"BS call: {bs_call:.4f}")
for steps_mc in steps_list:
    mc = MonteCarloPricing(S_0=S0, X=K, r=r, sigma=sigma, T=T,
                           num_paths=paths, steps=steps_mc, seed=123)
    mc_price, mc_se = mc.european(call=True, antithetic=False)
    diff = abs(mc_price - bs_call)
    print(f"steps={steps_mc:4d} MC={mc_price:.4f} SE={mc_se:.4f} |MC-BS|={diff:.4f}")


BS call: 10.4506
steps=   1 MC=10.4496 SE=0.0208 |MC-BS|=0.0009
steps=  52 MC=10.4663 SE=0.0209 |MC-BS|=0.0157
steps= 252 MC=10.4700 SE=0.0208 |MC-BS|=0.0194


KeyboardInterrupt: 